In [1]:
import sys, platform, subprocess

print("Python      :", sys.version.split()[0])
print("Platform    :", platform.platform())

try:
    import torch
    print("PyTorch     :", torch.__version__)
    print("CUDA (torch):", torch.version.cuda)
    if torch.cuda.is_available():
        p = torch.cuda.get_device_properties(0)
        print("GPU         :", p.name)
        print("Memori GPU  : %.1f GB" % (p.total_memory / 1024**3))
    else:
        print("GPU         : tidak tersedia -> cek Runtime > Change runtime type")
except ImportError:
    print("PyTorch belum terpasang")

print("\n--- Versi pustaka ---")
for pkg in ["transformers", "accelerate", "tokenizers", "sentence-transformers",
            "faiss-cpu", "pandas", "numpy", "scikit-learn", "scipy"]:
    try:
        from importlib.metadata import version
        print(f"{pkg:22}: {version(pkg)}")
    except Exception:
        print(f"{pkg:22}: belum terpasang")

try:
    from importlib.metadata import version
    v = tuple(int(x) for x in version("transformers").split(".")[:2])
    if v < (4, 51):
        print("\nPERINGATAN: transformers < 4.51, Qwen3 belum didukung -> perlu upgrade")
    else:
        print("\nOK: transformers cukup baru untuk Qwen3")
except Exception:
    pass

print("\n--- nvidia-smi ---")
print(subprocess.run(["nvidia-smi"], capture_output=True, text=True).stdout)

Python      : 3.13.16
Platform    : Linux-6.6.159+-x86_64-with-glibc2.39
PyTorch     : 2.14.1+cu130
CUDA (torch): 13.0
GPU         : Tesla T4
Memori GPU  : 14.6 GB

--- Versi pustaka ---
transformers          : 5.22.1
accelerate            : 1.15.0
tokenizers            : 0.23.3
sentence-transformers : 5.7.0
faiss-cpu             : belum terpasang
pandas                : 2.2.3
numpy                 : 2.4.6
scikit-learn          : 1.6.1
scipy                 : 1.16.3

OK: transformers cukup baru untuk Qwen3

--- nvidia-smi ---
Sun Nov 22 00:50:12 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage |

In [2]:
import time, torch
from huggingface_hub import model_info
from transformers import AutoTokenizer, AutoModelForCausalLM

MODEL_ID = "Qwen/Qwen3-4B-Instruct-2507"

# 1) Verifikasi identitas model di Hugging Face
info = model_info(MODEL_ID)
print("Model ID  :", info.id)
print("Commit    :", info.sha)
print("Task      :", info.pipeline_tag)
print("Tag       :", [t for t in info.tags if t.startswith("license")])

# 2) Muat tokenizer dan model
t0 = time.time()
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID, dtype=torch.float16, device_map="auto"
)
model.eval()
waktu_muat = time.time() - t0

# 3) Catat ringkasan
n_param = sum(p.numel() for p in model.parameters())
print(f"\nParameter     : {n_param/1e9:.2f} miliar")
print("Dtype         :", next(model.parameters()).dtype)
print("Perangkat     :", next(model.parameters()).device)
print(f"Memori GPU    : {torch.cuda.memory_allocated()/1024**3:.2f} GB terpakai")
print(f"Waktu muat    : {waktu_muat:.1f} detik")
print("Ukuran vocab  :", len(tokenizer))

Model ID  : Qwen/Qwen3-4B-Instruct-2507
Commit    : cdbee75f17c01a7cc42f958dc650907174af0554
Task      : text-generation
Tag       : ['license:apache-2.0']


config.json:   0%|          | 0.00/727 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/9.38k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json: reconstructing file:   0%|          |  0.00B / 11.4MB            

... (3 outputs left out)

Fetching 3 files:   0%|          | 0/3 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/238 [00:00<?, ?B/s]


Parameter     : 4.02 miliar
Dtype         : torch.float16
Perangkat     : cuda:0
Memori GPU    : 7.49 GB terpakai
Waktu muat    : 161.1 detik
Ukuran vocab  : 151669


In [3]:
import time, torch

pertanyaan = "Apa saja syarat membuat paspor baru bagi WNI dewasa?"

messages = [{"role": "user", "content": pertanyaan}]
teks = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
inputs = tokenizer(teks, return_tensors="pt").to(model.device)
n_masuk = inputs["input_ids"].shape[1]

def jawab(maks=256):
    torch.cuda.synchronize()
    t0 = time.time()
    with torch.no_grad():
        out = model.generate(**inputs, max_new_tokens=maks, do_sample=False)
    torch.cuda.synchronize()
    dt = time.time() - t0
    baru = out[0][n_masuk:]
    return tokenizer.decode(baru, skip_special_tokens=True), dt, len(baru)

for i in (1, 2):
    jwb, dt, n = jawab()
    print(f"=== Percobaan {i} ===")
    print(f"Waktu: {dt:.1f} detik | token baru: {n} | kecepatan: {n/dt:.1f} token/detik")
    if i == 1:
        print("\nPertanyaan:", pertanyaan)
        print("Jawaban   :", jwb, "\n")

print(f"Memori GPU puncak: {torch.cuda.max_memory_allocated()/1024**3:.2f} GB")

=== Percobaan 1 ===
Waktu: 17.3 detik | token baru: 256 | kecepatan: 14.8 token/detik

Pertanyaan: Apa saja syarat membuat paspor baru bagi WNI dewasa?
Jawaban   : Untuk membuat paspor baru bagi **Warga Negara Indonesia (WNI) dewasa**, ada beberapa **syarat dan prosedur** yang harus dipenuhi. Berikut adalah daftar **syarat utama** yang umum berlaku (sebagai referensi hingga tahun 2024):

---

### ✅ **Syarat Umum Membuat Paspor Baru bagi WNI Dewasa**

1. **Usia**
   - WNI usia **18 tahun ke atas** (dewasa).
   - WNI di bawah 18 tahun harus membuat paspor dengan izin orang tua/wali.

2. **KTP (Kartu Tanda Penduduk)**
   - Harus memiliki **KTP yang masih berlaku** (tidak kadaluarsa).
   - KTP harus diperbarui jika sudah kadaluarsa (dalam 6 bulan terakhir).

3. **Paspor Lama (jika ada)**
   - Jika memiliki paspor lama, maka harus **dibatalkan** (dibawa ke kantor imigrasi untuk penggantian).
   - Paspor lama harus **dib 

=== Percobaan 2 ===
Waktu: 14.4 detik | token baru: 256 | kecepatan: 